# Phase 5: LightGBM + SHAP

Same walk-forward protocol as Phase 4 (folds 2019-2024, each trained on earlier seasons only, 2025 untouched). Same six features as the final Phase 4 model (C6), same per-fold K picks. The only change is the model learning the adjustment on top of carry-over: a team of small trees instead of one weight per feature. Tree settings are picked per fold on season S−1. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [1]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import itertools
import polars as pl
from src.data.load import load_pbp
from src.features.qb_games import qb_games
from src.features.target import build_targets
from src.features.history import BASE_FEATURES, SACK_FEATURES, base_features, sack_features
from src.features.opponent import DEFENSE_FEATURES, defense_features
from src.models.baselines import league_average
from src.models.boosting import LGBM_GRID, make_lgbm
from src.models.evaluate import bootstrap_compare, mae
from src.models.walk_forward import Experiment, compare, walk_forward

# 2015 = history only; 2025 = final test, not loaded here.
games = qb_games(load_pbp(list(range(2015, 2025))))
targets = build_targets(games)

DEF_K_GRID = (50, 100, 200, 300, 500, 800, 1200, 2000)
SACK_K_GRID = (50, 100, 200, 300, 500, 800, 1200)
FEATURES = BASE_FEATURES + DEFENSE_FEATURES + SACK_FEATURES
build = lambda g, t, la, k: sack_features(g, defense_features(g, base_features(g, t, la), la, k[0]), k[1])
K_GRID = tuple(itertools.product(DEF_K_GRID, SACK_K_GRID))

## Concept 1: gradient boosting vs the final Phase 4 model

Keep rule as always: pooled bootstrap win ≥ 80% AND better in ≥ 4 of 6 seasons. Also split by history (last season ≤ 212 dropbacks, the first question a small tree asks) to see *where* any gain comes from.

In [2]:
c6, _ = walk_forward(games, targets, Experiment("C6", FEATURES, build, K_GRID))
lgbm, picks = walk_forward(games, targets, Experiment("LGBM", FEATURES, build, K_GRID, make_model=make_lgbm, param_grid=LGBM_GRID))
s = compare(lgbm, c6)
print(f"pooled MAE  C6 linear {s['best_mae']:.4f}   LightGBM {s['mae']:.4f}   win {100 * s['win_rate']:.1f}%   seasons won {s['seasons_won']}/6   keep {s['keep']}")
print("per-season gain:", {y: round(g, 4) for y, g in s["per_season_gain"].items()})
for p in picks:
    imp = p["model"].booster_.feature_importance("gain")
    print(p["season"], "(leaves, trees) =", p["param"], " share of gain:", {f: round(float(v / imp.sum()), 2) for f, v in zip(FEATURES, imp)})

hist = base_features(games, targets, league_average(targets)).select("passer_id", "season", "week", "last_dropbacks")
j = lgbm.join(c6.select("passer_id", "season", "week", pl.col("prediction").alias("c6")), on=["passer_id", "season", "week"]).join(hist, on=["passer_id", "season", "week"])
pl.DataFrame([{"group": label, "games": (x := j.filter(cond)).height, "C6": round(mae(x, "c6")["weighted"], 4),
               "LightGBM": round(mae(x)["weighted"], 4), "win %": round(100 * bootstrap_compare(x, "c6", "prediction")["win_rate"], 1)}
              for label, cond in [("low history (last season <= 212)", pl.col("last_dropbacks") <= 212), ("veterans", pl.col("last_dropbacks") > 212)]])

pooled MAE  C6 linear 0.2256   LightGBM 0.2254   win 67.6%   seasons won 3/6   keep False
per-season gain: {2019: -0.0004, 2020: -0.0007, 2021: 0.0005, 2022: -0.0003, 2023: 0.0009, 2024: 0.0012}
2019 (leaves, trees) = (4, 100)  share of gain: {'std_avg': 0.09, 'std_dropbacks': 0.24, 'last_avg': 0.07, 'last_dropbacks': 0.22, 'def_shrunk': 0.26, 'sack_carry': 0.13}
2020 (leaves, trees) = (4, 100)  share of gain: {'std_avg': 0.09, 'std_dropbacks': 0.16, 'last_avg': 0.07, 'last_dropbacks': 0.24, 'def_shrunk': 0.3, 'sack_carry': 0.14}
2021 (leaves, trees) = (4, 100)  share of gain: {'std_avg': 0.05, 'std_dropbacks': 0.14, 'last_avg': 0.09, 'last_dropbacks': 0.26, 'def_shrunk': 0.37, 'sack_carry': 0.09}
2022 (leaves, trees) = (8, 100)  share of gain: {'std_avg': 0.11, 'std_dropbacks': 0.16, 'last_avg': 0.09, 'last_dropbacks': 0.22, 'def_shrunk': 0.29, 'sack_carry': 0.13}
2023 (leaves, trees) = (4, 100)  share of gain: {'std_avg': 0.03, 'std_dropbacks': 0.15, 'last_avg': 0.07, 'last_dropbacks

group,games,C6,LightGBM,win %
str,i64,f64,f64,f64
"""low history (last season <= 21…",1271,0.2428,0.2417,83.9
"""veterans""",2411,0.218,0.2181,39.4


### ▶ Play: Concept 1
Fix the tree settings instead of letting each fold pick them.

In [3]:
# ---- KNOBS -----------------------------------------------------------------
LEAVES = 4    # Try 2 (one question per tree: no interactions at all) or 32 (deep trees: can they find more, or just flukes?)
TREES = 100   # Try 30 (barely moves off carry-over) or 1000 (keeps correcting long after the signal runs out)
# ---------------------------------------------------------------------------

play, _ = walk_forward(games, targets, Experiment("LGBM play", FEATURES, build, K_GRID, make_model=make_lgbm, param_grid=((LEAVES, TREES),)))
s = compare(play, c6)
print(f"pooled MAE  C6 {s['best_mae']:.4f}   LightGBM default {mae(lgbm)['weighted']:.4f}   now ({LEAVES} leaves, {TREES} trees) {s['mae']:.4f}   win {100 * s['win_rate']:.1f}%   seasons {s['seasons_won']}/6")

pooled MAE  C6 0.2256   LightGBM default 0.2254   now (4 leaves, 100 trees) 0.2252   win 80.8%   seasons 4/6


## Concept 2: SHAP for the final (linear) model

Every rating splits exactly into: carry-over + the average adjustment + one piece per feature, where each piece is `weight × (his value − training mean)`. That's what SHAP gives for any linear model; checked against `shap.LinearExplainer` below (the background is the fold's own training rows). Model: the 2024 fold, trained on 2016-2023.

In [4]:
import numpy as np
import shap
from src.models.baselines import carry_over
from src.models.final import FINAL_FEATURES, build_final, check_explanation_adds_up, explain, fit_and_predict

rows24, fitted24 = fit_and_predict(games, targets, 2024)
ex24 = explain(rows24, fitted24)
check_explanation_adds_up(ex24)

# The shap library's answer, with the fold's training rows (built exactly as the fold built them) as background.
all_rows = carry_over(games, targets, fitted24["league_avg"], fitted24["carry_k"]).rename({"prediction": "carry_over"})
X_train = build_final(games, all_rows, fitted24["league_avg"], fitted24["feature_k"]).filter(pl.col("season") < 2024).select(FINAL_FEATURES).to_numpy()
ref = shap.LinearExplainer(fitted24["model"], shap.maskers.Independent(X_train, max_samples=len(X_train))).shap_values(ex24.select(FINAL_FEATURES).to_numpy())
print("max difference vs shap library, all 2024 games:", np.abs(ref - ex24.select(f"{c}_contrib" for c in FINAL_FEATURES).to_numpy()).max())

max difference vs shap library, all 2024 games: 2.0816681711721685e-17


### ▶ Play: Concept 2
Explain any 2024 QB-game.

In [5]:
# ---- KNOBS -----------------------------------------------------------------
QB = "P.Mahomes"   # Try "C.Williams" (rookie: no last season) or "L.Jackson"
WEEK = 10          # Try 1 (nothing played yet this season) vs a late week
# ---------------------------------------------------------------------------

r = ex24.filter(pl.col("passer") == QB, pl.col("week") == WEEK)
if r.is_empty():
    print(f"{QB} has no 2024 game in week {WEEK}")
else:
    r = r.row(0, named=True)
    print(f"{QB} 2024 week {WEEK} vs {r['defteam']}")
    print(f"  carry-over               {r['carry_over']:+.3f}")
    print(f"  average adjustment       {r['avg_adjustment']:+.3f}")
    for c in FINAL_FEATURES:
        print(f"  {c:<15} = {r[c]:>8.3f}   {r[c + '_contrib']:+.3f}")
    print(f"  = rating                 {r['prediction']:+.3f}   (actual: {r['target']:+.3f})")

P.Mahomes 2024 week 10 vs DEN
  carry-over               +0.138
  average adjustment       -0.025
  std_avg         =    0.152   +0.003
  std_dropbacks   =  306.000   +0.003
  last_avg        =    0.163   +0.003
  last_dropbacks  =  839.000   +0.050
  def_shrunk      =   -0.062   -0.061
  sack_carry      =    0.052   -0.020
  = rating                 +0.092   (actual: +0.109)


## Your exploration

Add cells below.